# Phase 3a — Approach 1: Custom CNN from Scratch

**CSC3109 Machine Learning | Group 30**

## What Is This?

We build a Convolutional Neural Network (CNN) entirely from scratch — no pre-trained weights, no borrowed architecture. Every weight starts at a random value and is learned purely from our 2,400 training images.

**Role in the project:** This is our **baseline**. We do not expect it to be the best performer. Its purpose is to:
1. Prove we understand CNN fundamentals from first principles
2. Establish a performance floor to compare all other approaches against
3. Show what is achievable without transfer learning

## How a CNN Works (Brief)

A CNN learns to detect visual features in a hierarchy:
- **Early layers** → detect simple patterns (edges, corners)
- **Middle layers** → combine edges into textures and shapes
- **Late layers** → combine shapes into object parts
- **Final layer** → classify the full image into one of 4 categories

Each `Conv2D` layer slides a small filter (e.g. 3×3) across the image and learns which patterns activate it. `MaxPool` shrinks the spatial dimensions, reducing computation and forcing the model to focus on the most prominent features.

In [ ]:
import os
import sys
from pathlib import Path

# AUTO-DETECT LOCAL vs COLAB
if os.path.exists('/content'):
    REPO_ROOT = Path('/content/csc3109-g30')
    print('Detected: Colab environment')
else:
    REPO_ROOT = Path('..').resolve()
    print('Detected: Local environment')

sys.path.insert(0, str(REPO_ROOT / 'src'))

# Verify paths
assert (REPO_ROOT / 'src' / 'dataset.py').exists(), f"dataset.py not found at {REPO_ROOT / 'src'}"
assert (REPO_ROOT / 'data' / 'train').exists(), f"train data not found at {REPO_ROOT / 'data'}"

import torch
import torch.nn as nn
import torch.optim as optim

from dataset import get_data_loaders
from train_utils import train_model, plot_training_curves, plot_confusion_matrix, \
                        get_all_predictions, print_classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPORT_DIR = REPO_ROOT / 'report'
REPORT_DIR.mkdir(exist_ok=True)

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'Device: {DEVICE}')
print('All imports OK')

---
## Step 1 — Load Data

In [ ]:
train_loader, val_loader, cls2idx, idx2cls = get_data_loaders(
    data_dir=REPO_ROOT / 'data',
    batch_size=32,
    num_workers=0
)

NUM_CLASSES = len(cls2idx)
print(f'Classes: {cls2idx}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

---
## Step 2 — Define the Model Architecture

Our custom CNN has 3 convolutional blocks followed by a fully connected classifier.

```
Input: 224×224×3 image
  ↓
Conv Block 1: Conv(32 filters) → BatchNorm → ReLU → MaxPool  → 112×112×32
  ↓
Conv Block 2: Conv(64 filters) → BatchNorm → ReLU → MaxPool  →  56×56×64
  ↓
Conv Block 3: Conv(128 filters) → BatchNorm → ReLU → MaxPool →  28×28×128
  ↓
Conv Block 4: Conv(256 filters) → BatchNorm → ReLU → MaxPool →  14×14×256
  ↓
Adaptive Average Pool → 1×1×256 (i.e. 256 numbers)
  ↓
Linear(256) → ReLU → Dropout(0.5)
  ↓
Linear(4)  → raw scores for 4 classes
```

**BatchNorm:** normalises layer outputs to stabilise training — almost always used in modern CNNs.  
**Dropout:** randomly zeroes 50% of neurons during training to prevent overfitting.

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2, stride=2)  # halve spatial dimensions
            )

        self.features = nn.Sequential(
            conv_block(3,   32),   # 224 → 112
            conv_block(32,  64),   # 112 →  56
            conv_block(64,  128),  #  56 →  28
            conv_block(128, 256),  #  28 →  14
        )

        # AdaptiveAvgPool reduces any spatial size to exactly 1×1
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


model = CustomCNN(num_classes=NUM_CLASSES)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: CustomCNN')
print(f'Trainable parameters: {total_params:,}')
print(model)

---
## Step 3 — Configure Training

**Optimiser — Adam:**  
Adam (Adaptive Moment Estimation) adjusts the learning rate for each parameter individually. It is the most popular choice for starting out — reliable and requires less tuning than SGD.

**Learning rate (1e-3):**  
Controls how large each weight update is. Too high → training is unstable. Too low → training is very slow. 1e-3 is a good default for Adam.

**Scheduler — ReduceLROnPlateau:**  
If validation loss stops improving for 5 epochs (patience=5), the learning rate is reduced by 50%. This gives training a "second wind" when it gets stuck.

In [ ]:
NUM_EPOCHS = 30  # increase to 50 if you have time

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

print(f'Optimiser: Adam (lr=1e-3, weight_decay=1e-4)')
print(f'Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)')
print(f'Epochs: {NUM_EPOCHS}')
print()
print('Estimated time on CPU: ~2–5 min/epoch depending on hardware')
print('Tip: run on Google Colab GPU for 10x speedup')

---
## Step 4 — Train

This is the main training loop. Each epoch:
1. Feeds all training images through the model (forward pass)
2. Computes the loss (how wrong were the predictions?)
3. Computes gradients (which direction to adjust weights?)
4. Updates weights (optimizer.step)
5. Evaluates on validation set (no weight updates)
6. Saves checkpoint if validation accuracy improved

The best weights are automatically saved to `models/custom_cnn_best.pth`.

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    save_name='custom_cnn',
    device=DEVICE
)

---
## Step 5 — Plot Training Curves

**What to look for:**
- **Overfitting:** train accuracy much higher than val accuracy → model memorised training data
- **Underfitting:** both accuracies are low → model is too simple or not trained long enough
- **Good fit:** both curves close together and rising

In [ ]:
plot_training_curves(
    history,
    title='Approach 1 — Custom CNN Training Curves',
    save_path=REPORT_DIR / 'approach1_custom_cnn_curves.png'
)

---
## Step 6 — Evaluate on Validation Set

In [ ]:
all_labels, all_preds = get_all_predictions(model, val_loader, DEVICE)

print('=== Approach 1: Custom CNN — Validation Results ===')
print_classification_report(all_labels, all_preds)

overall_acc = (all_labels == all_preds).mean()
print(f'Overall Validation Accuracy: {overall_acc:.4f} ({overall_acc*100:.2f}%)')

In [ ]:
plot_confusion_matrix(
    all_labels, all_preds,
    title='Approach 1 — Custom CNN Confusion Matrix',
    save_path=REPORT_DIR / 'approach1_custom_cnn_confusion.png'
)

---
## Step 7 — Observations for Report

*Fill in after running training. Consider:*

- What was the final validation accuracy?
- Did the model overfit? (large gap between train and val accuracy?)
- Which categories were most confused?
- How does training from scratch compare to what you'd expect from transfer learning?

**Strengths of Custom CNN:**
- Architecture tailored to our specific task
- No dependency on ImageNet pre-training
- Lightweight and fast to run at inference time

**Weaknesses of Custom CNN:**
- Limited feature richness compared to models trained on millions of images
- Requires more epochs and data to converge
- Likely underperforms transfer learning approaches on this small dataset (700 images/class)

---
## Done
Best model saved to `models/custom_cnn_best.pth`

**Next:** `03b_resnet50_extraction.ipynb` — ResNet-50 Feature Extraction